# Watermarks, Lateness and the Completeness Trade

Companion notebook for the [Event Time, Watermarks and Windows](https://codemia.io/courses/big_data_engines/bd_event_time_watermarks_and_windows) lesson on Codemia.

**What you will build**: a small stream runtime, about forty lines of it, with event time windows and a watermark you
control. You will replay two hours of events whose arrival order is not their event order, sweep the
allowed lateness from zero to ten minutes, and plot the curve that every streaming job sits
somewhere on: how fresh the answer is against how complete it is. Then you will stall the watermark
with an idle source, which is the failure everybody meets in week two.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(23)

## Concept recap

Every streaming system has to answer one question that batch never asks: the window for 10:00 to
10:01 is over in wall clock terms, but is it over in event terms? Some of its events are still in
flight, in a mobile client's retry queue, in a Kafka partition a consumer is behind on, in a
rebalancing broker.

A watermark is the system's claim that no event older than a given event time will arrive any more.
It is a claim, not a fact. You set how conservative it is with an allowed lateness, and that single
number decides both how long you wait before answering and how much data you throw away. The rest
of this notebook measures both sides of it.

## Two hours of events that arrive out of order

In [ ]:
N = 200_000
SPAN = 7_200              # two hours of event time, in seconds
WINDOW = 60               # tumbling one minute windows

event_time = np.sort(rng.uniform(0, SPAN, N))

# Arrival delay: a fast majority, a slow minority on bad connections, and a thin tail of
# clients that were offline and flushed their queue when they came back.
delay = rng.lognormal(np.log(0.4), 0.6, N)
cls = rng.random(N)
mid = cls < 0.05
delay[mid] = rng.lognormal(np.log(8.0), 0.8, int(mid.sum()))
very_late = cls < 0.005
delay[very_late] = rng.uniform(60, 900, int(very_late.sum()))

arrival = event_time + delay

for q in [50, 90, 99, 99.9, 100]:
    print(f'delay p{q:<5} {np.percentile(delay, q):>8.1f} s')
print()
print(f'events whose delay exceeds one window ({WINDOW} s): '
      f'{(delay > WINDOW).mean():.2%}')
arrival_rank = np.argsort(np.argsort(arrival))
print(f'consecutive event pairs that arrive in the wrong order: '
      f'{(np.diff(arrival_rank) < 0).mean():.1%}')

Those percentiles are the shape of every real stream: the median event is here in well under a
second, and the decision you are about to make is governed entirely by the last tenth of a percent.
Note the gap between p99 and p99.9. If you size the lateness for the median you drop the tail, and
if you size it for the maximum your dashboard is fifteen minutes stale.

## A stream runtime in forty lines

In [ ]:
def run_stream(event_time, arrival, lateness, window=WINDOW, drop_late=True):
    """Process events in arrival order, advance a watermark, fire windows when it passes them.

    Returns per-window firing delay and completeness, plus the counts that matter.
    """
    order = np.argsort(arrival, kind='stable')
    et, at = event_time[order], arrival[order]

    true_counts = np.bincount((event_time // window).astype(int))
    counted = np.zeros(len(true_counts), dtype=int)
    fired = {}                      # window index -> (processing time, count at firing)

    max_seen = -np.inf
    dropped = 0
    updates = 0

    for i in range(len(et)):
        watermark = max_seen - lateness          # the claim, before this event is seen
        w = int(et[i] // window)

        if et[i] < watermark:                    # the window is already closed
            if drop_late:
                dropped += 1
            else:
                counted[w] += 1
                if w in fired:
                    updates += 1                 # a corrected result goes downstream
        else:
            counted[w] += 1

        max_seen = max(max_seen, et[i])
        watermark = max_seen - lateness
        for w_done in range(len(true_counts)):   # fire everything the watermark has passed
            if w_done not in fired and (w_done + 1) * window <= watermark:
                fired[w_done] = (at[i], counted[w_done])

    delays, completeness = [], []
    for w, (proc_time, count) in sorted(fired.items()):
        delays.append(proc_time - (w + 1) * window)
        completeness.append(count / true_counts[w])

    return {
        'lateness': lateness,
        'windows_fired': len(fired),
        'windows_total': len(true_counts),
        'dropped': dropped,
        'dropped_frac': dropped / len(et),
        'updates': updates,
        'median_fire_delay': float(np.median(delays)) if delays else float('nan'),
        'p99_fire_delay': float(np.percentile(delays, 99)) if delays else float('nan'),
        'mean_completeness': float(np.mean(completeness)) if completeness else float('nan'),
        'final_completeness': float(counted.sum() / true_counts.sum()),
        'worst_completeness': float(np.min(completeness)) if completeness else float('nan'),
    }


r = run_stream(event_time, arrival, lateness=5)
for k, v in r.items():
    print(f'{k:>20}  {v}')

One hundred and nineteen of the hundred and twenty windows fired. The last one never did, because
nothing arrives afterwards to push the watermark past its end, and that is not a quirk of this
simulation: a real job's most recent window is permanently unfired until the next event arrives.
Engines paper over it with a periodic watermark emitted on a timer, or with an end-of-stream marker
when the source is bounded. If you have ever watched a streaming dashboard whose newest bar is
always missing, this is why.

The inner loop is the whole semantics. An event is late if its event time is already behind the
watermark, and a window fires the moment the watermark passes its end. Nothing here knows about wall
clock time, which is why replaying a day of history through the same code produces the same answers
as the live stream did, and why event time is worth the trouble.

## The sweep: freshness against completeness

In [ ]:
LATENESS = [0, 1, 5, 15, 30, 60, 120, 300, 600]
runs = [run_stream(event_time, arrival, lateness=l) for l in LATENESS]

print(f"{'lateness':>9}{'dropped':>10}{'dropped %':>11}{'mean compl.':>13}"
      f"{'worst compl.':>14}{'fire delay s':>14}")
for r in runs:
    print(f"{r['lateness']:>9}{r['dropped']:>10,}{r['dropped_frac']:>10.3%}"
          f"{r['mean_completeness']:>13.4%}{r['worst_completeness']:>14.2%}"
          f"{r['median_fire_delay']:>14.1f}")

In [ ]:
dropped = [r['dropped_frac'] * 100 for r in runs]
fire = [r['median_fire_delay'] for r in runs]

plt.figure(figsize=(9, 5))
plt.plot(fire, dropped, marker='o', color='tab:purple')
for r, x, y in zip(runs, fire, dropped):
    plt.annotate(f"L={r['lateness']}s", (x, y), textcoords='offset points', xytext=(8, 6))
plt.xlabel('how long after the window closes the answer appears (seconds)')
plt.ylabel('events dropped as late (%)')
plt.title('Every streaming job lives on this curve. The question is where.')
plt.grid(alpha=0.3)
plt.show()

Read the curve as a menu rather than an optimisation problem, because there is no optimum in it. The
knee is where an extra second of waiting stops buying a meaningful amount of data, and after it you
are paying freshness for nothing. Where you sit depends on the consumer: a fraud model that acts in
seconds should take the drops, and a billing aggregate that will be invoiced should not drop
anything at all.

## The third option: fire early and correct

In [ ]:
early = run_stream(event_time, arrival, lateness=5, drop_late=False)
strict = run_stream(event_time, arrival, lateness=600)

print(f"{'strategy':>26}{'fire delay s':>14}{'dropped':>10}"
      f"{'first answer':>14}{'after fixes':>13}{'updates':>10}")
for name, r in [('L=5 s, drop late', runs[2]),
                ('L=600 s, drop late', strict),
                ('L=5 s, correct late', early)]:
    print(f'{name:>26}{r["median_fire_delay"]:>14.1f}{r["dropped"]:>10,}'
          f'{r["mean_completeness"]:>14.2%}{r["final_completeness"]:>13.2%}'
          f'{r["updates"]:>10,}')

Read the two completeness columns together. The first answer is what the consumer sees immediately,
and the corrected figure is what the window converges to once the stragglers land. The middle row
buys that convergence with ten minutes of staleness; the bottom row gets there in five seconds and
sends nearly two thousand corrections to do it.

The third row is why Dataflow's triggers, Flink's allowed lateness and the retract streams in a
streaming SQL engine exist. You get the fast answer and the right answer, and you pay in a different
currency: every correction is a downstream write, and the sink has to be able to absorb it. An
upsert into a key value store or a merge into an Iceberg table takes corrections happily. An
append-only ledger, an alerting webhook or a counter that somebody already screenshotted does not.

So the real question is not what lateness to set. It is whether the sink is idempotent, because that
is what decides which of the three rows you are allowed to choose.

## The failure everybody meets: an idle source

In [ ]:
# Source A is steady. Source B goes quiet between event time 1800 and 2700.
a_times = np.sort(rng.uniform(0, SPAN, 60_000))
b_times = np.sort(rng.uniform(0, SPAN, 20_000))
b_times = b_times[(b_times < 1800) | (b_times > 2700)]
two_source_stream = sorted([(t, 0) for t in a_times] + [(t, 1) for t in b_times])


def run_two_sources(lateness, idle_timeout=None, window=WINDOW):
    """Watermark across sources is the minimum of their watermarks, which is the trap."""
    events = two_source_stream
    per_source = [-np.inf, -np.inf]
    last_activity = [-np.inf, -np.inf]
    fired = {}
    stalls = []

    for et, src in events:                       # arrival order approximates event order here
        per_source[src] = et
        last_activity[src] = et

        active = [i for i in (0, 1)
                  if idle_timeout is None or et - last_activity[i] <= idle_timeout]
        watermark = min(per_source[i] for i in active) - lateness

        for w in range(SPAN // window):
            if w not in fired and (w + 1) * window <= watermark:
                fired[w] = et - (w + 1) * window

    for w in sorted(fired):
        stalls.append(fired[w])
    return np.array(stalls)


naive = run_two_sources(lateness=5)
with_timeout = run_two_sources(lateness=5, idle_timeout=120)

print(f'windows fired, no idle handling   {len(naive)}')
print(f'windows fired, 120 s idle timeout {len(with_timeout)}')
print()
print(f'worst firing delay, no idle handling   {naive.max():>8.0f} s')
print(f'worst firing delay, with idle timeout  {with_timeout.max():>8.0f} s')
print()
print('windows whose answer was more than two minutes late:')
print(f'  no idle handling   {(naive > 120).sum()}')
print(f'  with idle timeout  {(with_timeout > 120).sum()}')

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(np.arange(len(naive)) * WINDOW / 60, naive, label='watermark = min over all sources')
plt.plot(np.arange(len(with_timeout)) * WINDOW / 60, with_timeout,
         label='sources idle for 120 s are ignored')
plt.axvspan(30, 45, color='tab:orange', alpha=0.2, label='source B silent')
plt.xlabel('window, in minutes of event time')
plt.ylabel('seconds between window end and its answer')
plt.title('One quiet partition freezes the clock for the whole job')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The watermark across sources has to be the minimum, because any source could still deliver an old
event, and that is exactly what makes a quiet source so damaging: a partition with no traffic is
indistinguishable from a partition that is behind. Every engine ships the same workaround under a
different name, Flink's `withIdleness`, Kafka Streams' `max.idle.ms`, Dataflow's per key watermarks,
and every one of them trades correctness for liveness by guessing that silence means idleness.

This is also the shape of the bug when an over-provisioned topic has sixty-four partitions and
twelve of them never receive a key. The job looks healthy, lag is zero, and no window ever closes.

## What to take away

- A watermark is a guess about the future, and allowed lateness is how much you pay to make the
  guess safer.
- Drops are concentrated in the tail of the delay distribution, so the useful sweep is over
  percentiles of that distribution and not over round numbers.
- Firing early and correcting later gets both freshness and completeness, and is only available if
  the sink accepts updates. Decide that first.
- The watermark of a job is the minimum over its sources, so one silent partition stops time. Expect
  to configure idleness, and expect that configuration to be a correctness compromise.
- Window firing must depend on event time only. The moment wall clock leaks into the logic, a replay
  stops reproducing the original answers and your backfills quietly disagree with production.